# One acquisition loop, five versions: same sensor, different datasheet

A simulated soft sensor is indented by a simulated stage, and the same measurement is recorded by five
versions of the acquisition loop (V0 naive, V1 polling, V2 threaded, V3 frame-stamped, V4 double-rate tick).
Each version fixes the previous one's bug, and the datasheet you would quote changes with it. The demo uses real
threads, so it runs in real time (about 75 s for the fast run).

**Your numbers will differ from the slides: that is the point.** Thread wake-up jitter belongs to the host,
so the exact percentages move from machine to machine. The pattern (which version loses frames, which
hides it) does not.

Run the cells top to bottom. No uploads needed.

## 1. The code
Each cell writes one source file, byte for byte as `demo/<file>`. They are generated from `demo/*.py`; do not edit them here.

In [ ]:
%%writefile loops.py
"""One acquisition loop, version by version; each version fixes the previous one's bug.

Every version logs rows (t, force, sensor) and writes each row through rig.disk.
"""
import collections
import math
import queue
import threading
import time

import numpy as np


class RateLimiter:
    """Fixed-rate tick on perf_counter deadlines (framework core/timing.py)."""

    def __init__(self, rate_hz, origin=None):
        """origin: a tick lands on origin + n * period (perf_counter s); default: first tick now."""
        self.period, now = 1.0 / rate_hz, time.perf_counter()
        self.next = now if origin is None else origin + math.ceil((now - origin) / self.period) * self.period - self.period

    def wait(self, stop=None):
        self.next += self.period
        delay = self.next - time.perf_counter()
        if delay <= 0:  # overrun: skip whole periods, keep the phase
            self.next += math.floor(-delay / self.period) * self.period
        elif stop is None:
            time.sleep(delay)
        else:
            stop.wait(delay)


def v0_naive(rig, targets, dwell):
    """AIS SI Listing 1: blocking move, sleep through the dwell, one read pair per step."""
    log = []
    for z in targets:
        rig.stage.move_to(z)
        time.sleep(dwell)
        f = rig.load_cell.read()
        s, _, k = rig.sensor.read()  # serial.read(): blocks until the next 50 Hz frame
        row = (rig.world.now(), f, s, k)  # k: simulator frame index, never logged by V0-V2
        rig.disk.write(row[:3])
        log.append(row)
    rig.t_stop = rig.world.now()  # logging ends here: frames produced later are not lost frames
    return log


def v1_polling(rig, targets, dwell, rate_hz=50.0):
    """V0 without the blocking move and sleep: one thread reads every tick through ramps and dwells."""
    log, lim = [], RateLimiter(rate_hz)
    for z in targets:
        rig.stage.start(z)  # V0: move_to(z) blocked
        t_hold = None
        while t_hold is None or rig.world.now() < t_hold + dwell:  # V0: time.sleep(dwell)
            if t_hold is None and rig.stage.position() == z:
                t_hold = rig.world.now()
            f = rig.load_cell.read()
            s, _, k = rig.sensor.read()
            row = (rig.world.now(), f, s, k)
            rig.disk.write(row[:3])  # a disk stall still stops every read
            log.append(row)
            lim.wait()
    rig.t_stop = rig.world.now()
    return log


PHASE_MS = 4.0  # pinned start offset (ms) of the tick vs the frame clock: tick coincides with frame arrival -> loss peaks (host-dependent)


def stamped_on_read(now, forces, sensor):
    """V2 row: both channels stamped at the tick; the sensor value is up to a frame + latency old.

    Lag sign: positive = sensor channel older than the force channel.
    """
    return now, forces[-1][1], sensor[0], sensor[2]


def stamped_at_frame(now, forces, sensor):
    """V3 row: stamped at the sensor frame time (+ frame seq), paired with the force interpolated at that time."""
    t_frame = sensor[1]
    ts, fs = zip(*forces)
    return t_frame, float(np.interp(t_frame, ts, fs)), sensor[0], sensor[2]


def threaded(rig, targets, dwell, make_row, rate_hz=50.0, log_in_dwell=True, phase_ms=PHASE_MS, log_seq=False):
    """Device pollers -> latest-value registers; fixed-rate coordinator -> FIFO -> logger thread.

    phase_ms: coordinator ticks fall on world.t0 + phase + n/rate; sensor frames on n * 20 ms (+ 4 ms latency).
    log_seq: the frame seq is a logged column (V3+); else it stays internal, for the true-loss count.
    log_in_dwell=False is the planted defect for check-the-checker.
    """
    stop, dwelling, fifo, log, lock = threading.Event(), threading.Event(), queue.Queue(), [], threading.Lock()
    forces = collections.deque(maxlen=50)  # (t, force) history at 1 kHz
    reg = {"sensor": (0.0, 0.0, 0)}  # latest (value, frame_time, frame index); a single assignment is GIL-atomic

    def poll_load_cell():
        lim = RateLimiter(1000.0)
        while not stop.is_set():
            with lock:
                forces.append((rig.world.now(), rig.load_cell.read()))
            lim.wait(stop)

    def poll_sensor():
        while not stop.is_set():
            reg["sensor"] = rig.sensor.read()

    def coordinate():
        lim, t_last, t_prev = RateLimiter(rate_hz, rig.world.t0 + phase_ms * 1e-3), -1.0, 0.0
        while not stop.is_set():
            t_tick = rig.world.now()
            with lock:
                history = list(forces)
            if history and (log_in_dwell or not dwelling.is_set()):
                row = make_row(rig.world.now(), history, reg["sensor"])
                if row[0] > t_last:  # one row per timestamp (V3: per new frame)
                    fifo.put(row)
                    t_last = row[0]
            # a frame counts as producible-while-logging once one full tick has passed since it arrived: a frame landing
            # on the last tick races the sensor thread's wake-up and cannot be logged
            rig.t_stop, t_prev = t_prev, t_tick
            lim.wait(stop)

    def write():
        for row in iter(fifo.get, None):
            rig.disk.write(row if log_seq else row[:3])
            log.append(row)

    workers = [threading.Thread(target=f, daemon=True) for f in (poll_load_cell, poll_sensor, coordinate)]
    writer = threading.Thread(target=write, daemon=True)
    for th in workers + [writer]:
        th.start()
    for z in targets:
        t_ramp_end = rig.stage.start(z)  # non-blocking, as in V1
        time.sleep(max(t_ramp_end - rig.world.now(), 0.0))
        dwelling.set()
        time.sleep(dwell)
        dwelling.clear()
    stop.set()
    for th in workers:
        th.join()
    fifo.put(None)
    writer.join()
    return log


def v2_threaded(rig, targets, dwell, **kw):
    """V1 + threads only: each device polled at its native rate, the disk behind a FIFO; rows stamped on read."""
    return threaded(rig, targets, dwell, stamped_on_read, **kw)


def v3_frame_stamped(rig, targets, dwell, **kw):
    """V2 with rows stamped at the sensor frame time."""
    return threaded(rig, targets, dwell, stamped_at_frame, **{"log_seq": True, **kw})


def v4_double_tick(rig, targets, dwell, **kw):
    """V3 + coordinator tick at 2x the frame rate, so every frame is seen whatever the phase."""
    return threaded(rig, targets, dwell, stamped_at_frame, **{"rate_hz": 100.0, "log_seq": True, **kw})


VERSIONS = {"V0": v0_naive, "V1": v1_polling, "V2": v2_threaded, "V3": v3_frame_stamped, "V4": v4_double_tick}


In [ ]:
%%writefile metrics.py
"""One datasheet function applied identically to every log (naive, threaded, ground truth)."""
import numpy as np

NAN = float("nan")


def holds(ramps, t_end):
    """Dwell windows: from the end of each ramp to the start of the next (last one to t_end)."""
    ends = [r[0] for r in ramps[1:]] + [t_end]
    return [(r[1], e) for r, e in zip(ramps, ends)]


def _in_ramps(t, ramps):
    return np.any([(t >= a) & (t <= b) for a, b, _ in ramps], axis=0)


def _direction(t, ramps):
    """+1 loading / -1 unloading: direction of the latest ramp started at or before t (0 before any)."""
    starts = np.array([r[0] for r in ramps])
    dirs = np.array([r[2] for r in ramps])
    idx = np.searchsorted(starts, t, side="right") - 1
    return np.where(idx >= 0, dirs[np.clip(idx, 0, None)], 0)


def _hysteresis_pct(force, sensor, direction, band, n_dwell):
    """Max gap between quadratic fits of loading vs unloading branches, % of sensor span."""
    if n_dwell < 10:  # too few points per step to fit a branch
        return NAN
    fits = []
    for d in (1, -1):
        m = band & (direction == d)
        if m.sum() < 3:
            return NAN
        fits.append((np.polyfit(force[m], sensor[m], 2), force[m].min(), force[m].max()))
    lo, hi = max(f[1] for f in fits), min(f[2] for f in fits)
    if hi <= lo:
        return NAN
    grid = np.linspace(lo, hi, 100)
    gap = np.abs(np.polyval(fits[0][0], grid) - np.polyval(fits[1][0], grid))
    return float(100 * gap.max() / np.ptp(sensor))


def frames_lost_pct(seq, n_frames=None):
    """% of frames never in `seq` (frame indices >= 1). n_frames: frames produced (simulator truth);
    default: the span first..last seen (all a log can know). NaN without a seq column."""
    if seq is None:
        return NAN
    k = np.unique(np.asarray(seq, int))
    k = k[(k >= 1) & (k <= (n_frames or k[-1]))]
    return float(100 * (1 - len(k) / (n_frames or k[-1] - k[0] + 1)))


def datasheet(t, force, sensor, ramps, t_end):
    """ramps = [(t_start, t_end, direction)] from the stage; t_end = end of protocol."""
    t, force, sensor = (np.asarray(a, float) for a in (t, force, sensor))
    dt_ms = np.diff(t) * 1e3
    fs = force.max()
    hold_masks = [(t > a) & (t <= b) for a, b in holds(ramps, t_end)]
    # unmeasured (< 2 samples) in any hold -> NaN: relaxation cannot be read off one point
    relax = [abs(force[m][0] - force[m][-1]) / fs if m.sum() >= 2 else NAN for m in hold_masks]
    band = (force >= 0.1 * fs) & (force <= 0.9 * fs)
    dwell_min = int(min(m.sum() for m in hold_masks))
    return {
        "samples": len(t),
        "rate_Hz": (len(t) - 1) / (t[-1] - t[0]),
        "dt_p50_ms": float(np.percentile(dt_ms, 50)),
        "dt_p99_ms": float(np.percentile(dt_ms, 99)),
        "dt_max_ms": float(dt_ms.max()),
        "ramp_samples": int(_in_ramps(t, ramps).sum()),
        "dwell_samples_min": dwell_min,
        "force_fs_N": float(fs),
        "relaxation_pct_fs": float(100 * np.mean(relax)),
        "sensitivity_V_per_N": float(np.polyfit(force[band], sensor[band], 1)[0]) if band.sum() >= 2 else NAN,
        "hysteresis_pct": _hysteresis_pct(force, sensor, _direction(t, ramps), band, dwell_min),
    }


def lag_ms(t, force, sensor, truth, ramps, max_ms=60.0):
    """Sensor-minus-force timestamp error (ms; positive = sensor older than force): each channel's delay vs the noiseless truth
    (columns t, force, sensor), least squares over ramp samples; NaN without >= 10 ramp samples."""
    t = np.asarray(t, float)
    m = _in_ramps(t, ramps)
    if m.sum() < 10:
        return NAN
    delays = np.arange(-max_ms, max_ms + 0.25, 0.5) * 1e-3

    def delay(y, col):
        sse = [np.sum((np.asarray(y)[m] - np.interp(t[m] - d, truth[:, 0], truth[:, col])) ** 2) for d in delays]
        return delays[int(np.argmin(sse))]

    return float(1e3 * (delay(sensor, 2) - delay(force, 1)))


def fmt(x):
    return f"{x:.4g}" if np.isfinite(x) else "n/a"


def _rel_err(a, b):
    return abs(a - b) / abs(b) if np.isfinite(a) else np.inf


def failures(ds, truth, period_ms):
    """What is wrong with a log: the name of each check it fails against the noiseless truth."""
    checks = {
        "no ramps": ds["ramp_samples"] > 0,
        "<10 dwell samples": ds["dwell_samples_min"] >= 10,
        "jitter p99>2T": ds["dt_p99_ms"] <= 2 * period_ms + 1e-6,  # tolerance: 40 ms gaps differ from 40.0 by float rounding
        "frames lost>1%": ds["frames_lost_true_pct"] <= 1.0,  # true loss: the simulator knows every frame
        "lag>5ms": abs(ds["sensor_force_lag_ms"]) <= 5.0,
        "full-scale>5%": _rel_err(ds["force_fs_N"], truth["force_fs_N"]) <= 0.05,
        "relaxation>20%": _rel_err(ds["relaxation_pct_fs"], truth["relaxation_pct_fs"]) <= 0.2,
        "sensitivity>5%": _rel_err(ds["sensitivity_V_per_N"], truth["sensitivity_V_per_N"]) <= 0.05,
        # hysteresis_pct is not checked: noise-level (0.5-1.3 % across seeds, full length), see README
    }
    return [name for name, ok in checks.items() if not ok]


In [ ]:
%%writefile plots.py
"""Vector figures for the projector: timeline, sensor-vs-force loops, datasheet table."""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

import metrics

plt.rcParams.update({"font.size": 18, "lines.linewidth": 2})
COLORS = {"V0": "tab:red", "V1": "tab:green", "V2": "tab:orange", "V3": "tab:blue", "V4": "tab:purple"}


def _style(v):
    big = v == "V0"  # naive has ~8 points: draw them large
    return {"color": COLORS.get(v, "k"), "marker": "o" if big else ".", "ms": 10 if big else 4, "ls": "none"}


def _save(fig, out, stem):
    fig.tight_layout()
    for ext in ("svg", "pdf"):
        fig.savefig(out / f"{stem}.{ext}")
    plt.close(fig)


def save_all(out, runs, sheets, truth, bugs):
    """runs: version -> (rig, log); sheets: version -> (datasheet, truth); bugs: version -> failed checks."""
    fig, axes = plt.subplots(len(runs), 1, figsize=(14, 3 * len(runs) + 1), sharex=True, sharey=True,
                             squeeze=False)
    for ax, (v, (rig, log)) in zip(axes[:, 0], runs.items()):
        tr, lg = np.array(rig.world.truth), np.array(log)
        for a, b, _ in rig.world.ramps:
            ax.axvspan(a, b, color="0.85")
        ax.plot(tr[:, 0], tr[:, 1], color="0.3", lw=1.5)
        ax.plot(lg[:, 0], lg[:, 1], label=f"{v} ({len(lg)} samples)", **_style(v))
        ax.set_ylabel("force (N)")
        ax.legend(loc="upper right")
    axes[-1, 0].set_xlabel("time (s)   [line = true force, grey = stage ramps]")
    _save(fig, out, "timeline")

    fig, ax = plt.subplots(figsize=(10, 8))
    tr = np.array(next(iter(runs.values()))[0].world.truth)
    ax.plot(tr[:, 1], tr[:, 2], color="0.3", lw=1.5, label="truth (noiseless 1 kHz)")
    for v, (_, log) in reversed(list(runs.items())):
        lg = np.array(log)
        ax.plot(lg[:, 1], lg[:, 2], label=v, **_style(v))
    ax.set(xlabel="force (N)", ylabel="sensor (V)")
    ax.legend()
    _save(fig, out, "loops")

    fig, ax = plt.subplots(figsize=(18, 8))
    ax.axis("off")
    rows = [[metrics.fmt(truth[k])] + [metrics.fmt(ds[k]) for ds, _ in sheets.values()] for k in truth]
    rows.append([""] + [bugs[v].replace(", ", "\n") for v in sheets])
    tab = ax.table(cellText=rows, rowLabels=[*truth, "bug exposed"], colLabels=["truth", *sheets], loc="center")
    tab.auto_set_font_size(False)
    tab.set_fontsize(14)
    tab.scale(1, 1.5)
    for c in range(len(sheets) + 1):
        tab[len(rows), c].set_height(tab[len(rows), c].get_height() * 6)
    _save(fig, out, "datasheet")


In [ ]:
%%writefile naive_vs_threaded.py
"""Same simulated soft sensor, one acquisition loop in five versions (loops.py), five datasheets.

V0 naive (AIS SI Listing 1) -> V1 single-thread polling -> V2 threaded, stamped on read
-> V3 threaded, stamped at frame time -> V4 coordinator at 2x frame rate. The "bug exposed" row lists the checks each version still fails.
Run:  uv run --with numpy --with matplotlib python naive_vs_threaded.py --fast --seed 1
"""
import argparse
import math
import threading
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np

import loops
import metrics
import plots

OUT = Path(__file__).resolve().parent / "out"


def sensor_curve(f):
    """Static soft-sensor transfer (V vs N), mildly saturating."""
    return 0.5 * f - 0.02 * f * f


class World:
    """Stage + standard-linear-solid sample + lagged sensor, integrated on demand at 1 kHz (= truth)."""

    def __init__(self, k_inf=1.0, k1=1.0, tau_relax=1.5, tau_sensor=0.05, dt=1e-3):
        self.k_inf, self.k1, self.tau_r, self.tau_s, self.dt = k_inf, k1, tau_relax, tau_sensor, dt
        self.y = self.f_lag = self.t = 0.0
        self.move = (0.0, 0.0, 0.0, 1.0)  # t0, x0, target, velocity
        self.ramps, self.truth = [], [(0.0, 0.0, 0.0)]
        self.lock = threading.Lock()
        self.t0 = time.perf_counter()

    def now(self):
        return time.perf_counter() - self.t0

    def _x(self, t):
        t0, x0, xt, v = self.move
        step = v * max(t - t0, 0.0)
        return xt if step >= abs(xt - x0) else x0 + math.copysign(step, xt - x0)

    def _advance(self):
        end = self.now()
        while self.t + self.dt <= end:
            self.t += self.dt
            x = self._x(self.t)
            self.y += (x - self.y) * self.dt / self.tau_r  # dashpot arm relaxes
            f = max(self.k_inf * x + self.k1 * (x - self.y), 0.0)  # probe cannot pull
            self.f_lag += (f - self.f_lag) * self.dt / self.tau_s
            self.truth.append((self.t, f, sensor_curve(self.f_lag)))
        return self.truth[-1]

    def sample(self):
        """(t, force, sensor) noiseless, at the current time."""
        with self.lock:
            return self._advance()

    def value_at(self, t):
        """Noiseless (t, force, sensor) at a past time t (1 ms grid)."""
        with self.lock:
            self._advance()
            return self.truth[min(max(round(t / self.dt), 0), len(self.truth) - 1)]

    def position(self):
        return self._x(self.now())

    def start_move(self, target, v):
        with self.lock:
            self._advance()
            t0 = self.now()  # same clock as log timestamps
            x0 = self._x(t0)
            t1 = t0 + abs(target - x0) / v
            self.move = (t0, x0, target, v)
            self.ramps.append((t0, t1, 1 if target > x0 else -1))
            return t1


class Stage:
    def __init__(self, world, v):
        self.world, self.v = world, v

    def start(self, z):
        """Non-blocking move command."""
        return self.world.start_move(z, self.v)

    def position(self):
        return self.world.position()

    def move_to(self, z):
        """Blocks until the ramp ends, like a real stage driver's move_to."""
        t1 = self.start(z)
        time.sleep(max(t1 - self.world.now(), 0.0))


class LoadCell:
    def __init__(self, world, rng, noise=0.02):
        self.world, self.rng, self.noise = world, rng, noise

    def read(self):
        return self.world.sample()[1] + self.rng.normal(0, self.noise)


class SoftSensor:
    """Samples a frame every `period` s; each frame arrives `latency` s later. Returns (value, frame_time, frame index)."""

    def __init__(self, world, rng, period=0.02, latency=0.004, noise=0.005):
        self.world, self.rng, self.period, self.latency, self.noise = world, rng, period, latency, noise

    def read(self):
        """Blocks until the next frame arrives (serial.read()); returns (value, frame_time, frame index)."""
        now = self.world.now()
        k = math.floor((now - self.latency) / self.period) + 1
        time.sleep(max(k * self.period + self.latency - now, 0.0))
        t_frame = k * self.period
        return self.world.value_at(t_frame)[2] + self.rng.normal(0, self.noise), t_frame, k


class Disk:
    """CSV writer with occasional OS stalls of tens of ms."""

    def __init__(self, path, rng, p_stall=0.05, stall_s=(0.02, 0.04)):
        self.f, self.rng, self.p, self.stall = open(path, "w"), rng, p_stall, stall_s  # noqa: SIM115 closed in run()
        self.cols = 0  # columns logged (3, or 4 with the frame seq)

    def write(self, row):
        if not self.cols:
            self.cols = len(row)
            self.f.write(",".join(("t_s", "force_N", "sensor_V", "seq")[:self.cols]) + "\n")
        self.f.write(",".join(f"{v:.6f}" for v in row) + "\n")
        self.f.flush()
        if self.rng.random() < self.p:
            time.sleep(self.rng.uniform(*self.stall))


def build_rig(name, seed, v, tau_relax, tau_sensor, p_stall=0.05, out=OUT):
    rngs = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(3)]
    world = World(tau_relax=tau_relax, tau_sensor=tau_sensor)
    out.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(world=world, stage=Stage(world, v), load_cell=LoadCell(world, rngs[0]),
                           sensor=SoftSensor(world, rngs[1]), disk=Disk(out / f"{name}_log.csv", rngs[2], p_stall))


def run(loop, rig, targets, dwell, **kw):
    """Run one loop version to completion; records the protocol end time and closes the log file."""
    log = loop(rig, targets, dwell, **kw)
    rig.world.sample()  # integrate truth to the end of the run
    rig.t_end = rig.world.now()  # protocol end (dwell windows); frame counting uses rig.t_stop, set by the loop
    rig.disk.f.close()
    return log


def evaluate(rig, log):
    """(datasheet of the log, datasheet of the same run's noiseless 1 kHz truth)."""
    truth_arr = np.array(rig.world.truth)
    t, f, s, k = np.array(log).T
    ds = metrics.datasheet(t, f, s, rig.world.ramps, rig.t_end)
    n_produced = math.floor((rig.t_stop - rig.sensor.latency) / rig.sensor.period)
    ds["frames_lost_true_pct"] = metrics.frames_lost_pct(k, n_produced)  # k: simulator's frame index
    ds["frames_lost_visible_pct"] = metrics.frames_lost_pct(k if rig.disk.cols == 4 else None)  # only if logged
    ds["sensor_force_lag_ms"] = metrics.lag_ms(t, f, s, truth_arr, rig.world.ramps)
    truth = metrics.datasheet(*truth_arr.T, rig.world.ramps, rig.t_end)
    truth.update(sensor_force_lag_ms=0.0, frames_lost_true_pct=0.0, frames_lost_visible_pct=0.0)
    return ds, truth


def protocol(fast):
    """Indentation staircase 0 -> top -> 0 mm; --fast: 1 mm steps to 3 mm, else 0.5 mm steps to 4 mm."""
    step, top = (1.0, 3.0) if fast else (0.5, 4.0)
    up = list(np.arange(step, top + 1e-9, step))
    return up + up[-2::-1] + [0.0]


def sweep(a):
    """Chosen threaded versions at several start phases: the same code, a different datasheet. Short protocol: 2 steps."""
    targets, dwell = (protocol(False) if not a.fast else [1.0, 0.0]), (a.dwell if not a.fast else 0.5)
    print(f"phase sweep, seed {a.seed}: coordinator tick offset vs sensor frame clock (frame arrives at +4 ms)")
    print(f"{'version':>8s}{'phase_ms':>9s}{'lost_true_%':>13s}{'lost_visible_%':>16s}{'lag_ms':>9s}")
    for v in a.sweep_versions.split(","):
        for ph in range(0, 20, 2):
            rig = build_rig("sweep", a.seed, a.velocity, a.tau_relax, a.tau_sensor, out=a.out)
            ds, _ = evaluate(rig, run(loops.VERSIONS[v], rig, targets, dwell, phase_ms=ph))
            print(f"{v:>8s}{ph:9d}" + "".join(f"{metrics.fmt(ds[k]):>{w}s}" for k, w in (
                ("frames_lost_true_pct", 13), ("frames_lost_visible_pct", 16), ("sensor_force_lag_ms", 9))))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--fast", action="store_true", help="6 steps (~15 s per version) instead of 16 (~40 s)")
    ap.add_argument("--versions", default=",".join(loops.VERSIONS), help="comma list, e.g. V0,V3")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--velocity", type=float, default=2.0, help="stage ramp speed, mm/s")
    ap.add_argument("--dwell", type=float, default=2.0, help="hold per step, s (Listing 1: 2.0)")
    ap.add_argument("--tau-relax", type=float, default=1.5, help="sample relaxation time constant, s")
    ap.add_argument("--tau-sensor", type=float, default=0.05, help="sensor first-order lag, s")
    ap.add_argument("--phase-ms", type=float, default=loops.PHASE_MS, help="V2-V4 coordinator tick offset vs frame clock")
    ap.add_argument("--phase-sweep", action="store_true", help="run --sweep-versions at phases 0..18 ms and print loss and lag")
    ap.add_argument("--sweep-versions", default="V2,V3", help="threaded versions for --phase-sweep")
    ap.add_argument("--out", type=Path, default=OUT, help="output folder for logs and plots (default: demo/out)")
    ap.add_argument("--no-stalls", action="store_true", help="disk never stalls (isolates the stall-driven V1 jitter)")
    a = ap.parse_args()
    if a.phase_sweep:
        return sweep(a)
    runs, sheets = {}, {}
    for v in a.versions.split(","):
        kw = {"phase_ms": a.phase_ms} if v in ("V2", "V3", "V4") else {}
        rig = build_rig(v, a.seed, a.velocity, a.tau_relax, a.tau_sensor, 0.0 if a.no_stalls else 0.05, a.out)
        t0 = time.perf_counter()
        runs[v] = (rig, run(loops.VERSIONS[v], rig, protocol(a.fast), a.dwell, **kw))
        sheets[v] = evaluate(*runs[v])
        print(f"{v} {loops.VERSIONS[v].__name__}: {time.perf_counter() - t0:.1f} s wall")
    truth = sheets[v][1]
    print(f"\ncoordinator phase (V2-V4): {a.phase_ms:g} ms after the frame clock origin")
    print(f"{'metric':22s}{'truth':>10s}" + "".join(f"{c:>10s}" for c in sheets))
    for k in truth:
        print(f"{k:22s}{metrics.fmt(truth[k]):>10s}" + "".join(f"{metrics.fmt(d[k]):>10s}" for d, _ in sheets.values()))
    print("hysteresis_pct: noise-level, not checked")
    bugs = {v: ", ".join(metrics.failures(ds, tr, 20.0)) or "none" for v, (ds, tr) in sheets.items()}
    print("bug exposed (failed checks vs truth):")
    for v, b in bugs.items():
        print(f"  {v}: {b}")
    plots.save_all(a.out, runs, sheets, truth, bugs)
    print(f"plots -> {a.out}")


if __name__ == "__main__":
    main()


## 2. Run all five versions (`--fast`, about 75 s)
Prints the datasheet table (truth vs V0-V4) and the checks each version still fails, then shows the plots.

In [ ]:
!python naive_vs_threaded.py --fast --seed 1 --out out
from IPython.display import SVG, display
for name in ("timeline", "loops", "datasheet"):
    display(SVG(filename=f"out/{name}.svg"))

## 3. Phase sweep
Frame loss depends on where the 50 Hz coordinator tick falls relative to the 50 Hz sensor frame clock.
`--phase-sweep --fast` runs V2, V3 and V4 at phases 0 to 18 ms (about 2.5 minutes in total). Look at true vs visible loss:
V2 loses frames its log cannot show; V3 shows its loss; V4 stays near zero at every phase.

In [ ]:
!python naive_vs_threaded.py --phase-sweep --sweep-versions V2,V3,V4 --fast --seed 1 --out out

## 4. Try this
Change one thing and compare with the run above:
- `--phase-ms 14` moves the tick away from the worst phase (4 ms).
- `--no-stalls` removes the simulated disk stalls (they cause V1's jitter).
- `--seed 2` draws different noise.
- `--versions V2,V3` runs only those versions.

In [ ]:
!python naive_vs_threaded.py --fast --versions V1,V2,V3 --phase-ms 14 --no-stalls --seed 2 --out out_try

## 5. Your host
Print this next to your numbers when you compare with someone else's.

In [ ]:
import os, platform
print(platform.platform())
print(platform.python_implementation(), platform.python_version())
print("CPUs:", os.cpu_count())